#SimCLR

In [ ]:
import torch #core PyTorch library
import torch.nn as nn #PyTorch's neural network building blocks
import torch.nn.functional as f #lower-level and faster functions
import torchvision
import torchvision.transforms as transforms #used for image preprocessing and augmentation
from torchvision import models #used to predefine CNN architectures
from torch.utils.data import DataLoader, Dataset

In [ ]:
class SimCLRTransform:
  def __init__(self, image_size=224):
    self.transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

  def __call__(self, image):
    return self.transform(image), self.transform(image) #return two independently augmented versions of the input image

In [ ]:
from PIL import Image
import os

class SimCLRDataset(Dataset):
  def __init__(self, path, transform):
    self.path = path
    self.transform = transform
    self.samples = [
        os.path.join(root, filename) #combines directory paths and file names into a full file path
        for root, _, files in os.walk(path) #walk through all folders and subfolders inside the given path but we ignore the subfolders
        for filename in files if filename.endswith(".jpg") #only takes .jpg files
    ]

  def __len__(self):
    return len(self.samples) #shows how many samples

  def __getitem__(self, idx):
    image_path = self.samples[idx]
    image = Image.open(image_path).convert("RGB") #get the image using index
    x_i, x_j = self.transform(image) #transform the image into two augmented images x_i and x_j
    return x_i, x_j

In [ ]:
class SimCLRModel(nn.Module):
  def __init__(self, projection_dim=128):
    super(SimCLRModel, self).__init__()

    resnet = models.resnet18(weights=None) #train from scratch
    self.encoder = nn.Sequential(*list(resnet.children())[:-1]) #use ResNet-18 as encoder(all layers except the last fully connected layer)
    self.flatten = nn.Flatten()

    #Multi-Layer Perceptron(MLP), mapping features to a contrastive space
    self.projector = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, projection_dim) #project encoder output into lower dimensional space, making it easier to learn contrastive similarities
    )

  def forward(self, x):
    h = self.encoder(x)
    h = self.flatten(h)
    z = self.projector(h)
    return z

In [ ]:
class NTXentLoss(nn.Module):
  def __init__(self, temperature=0.5):
    super(NTXentLoss, self).__init__()
    self.temperature = temperature #define the sharpness of contrast(lower = harder)

  def forward(self, z_i, z_j):

    #normalize(require for cosine similarity)
    z_i = f.normalize(z_i, dim=1)
    z_j = f.normalize(z_j, dim=1)

    z = torch.cat([z_i, z_j], dim=0) #combine two views
    similarity = torch.matmul(z, z.transpose(0, 1)) / self.temperature #each row in similarity is compared to all others(including itself which we don't want it to happen)
    batch_size = z_i.size(0)
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device) #create an identity matrix, which each row has a 1 on the diagonal
    similarity = similarity.masked_fill(mask, float('-inf')) #set all diagonal elements in similarity to -inf to prevent it to compare with itself

    positives = torch.cat([
        torch.arange(batch_size, 2 * batch_size),
        torch.arange(0, batch_size)
    ]).to(z.device) #tells the loss function the correct match

    loss = f.cross_entropy(similarity, positives)
    return loss

In [ ]:
from torch import optim
from tqdm import tqdm

epochs =200
batch_size = 128
temperature = 0.5
learning_rate = 3e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = SimCLRTransform(image_size=224)
dataset = SimCLRDataset(path="/content/drive/MyDrive/hw3/jpg", transform=transform) #change to the path where your dataset is in
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True) #dataset might not be divisable by 128 so we drop the last batch that smaller than 128
model = SimCLRModel(projection_dim=128).to(device)
loss_function = NTXentLoss(temperature=temperature)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
torch.cuda.empty_cache()

for epoch in range(1, epochs + 1):
  total_loss = 0
  model.train() #start train mode

  for x_i, x_j in tqdm(loader, desc=f"Epoch {epoch}/{epochs}"): #load a batch of paired augmented images
    x_i, x_j = x_i.to(device), x_j.to(device)

    z_i = model(x_i)
    z_j = model(x_j)
    loss = loss_function(z_i, z_j) #compute contrastive loss for two versions
    optimizer.zero_grad() #clear previous gradients
    loss.backward() #compute new gradients(direction of reducing the loss)
    optimizer.step()#update
    total_loss += loss.item()

  average_loss = total_loss / len(loader)
  print(f"Epoch {epoch}: Average Loss = {average_loss:.4f}")

Epoch 1/200: 100%|██████████| 63/63 [05:20<00:00,  5.09s/it]


Epoch 1: Average Loss = 4.9432


Epoch 2/200: 100%|██████████| 63/63 [03:42<00:00,  3.53s/it]


Epoch 2: Average Loss = 4.7316


Epoch 3/200: 100%|██████████| 63/63 [03:40<00:00,  3.50s/it]


...

Epoch 198: Average Loss = 3.8690


Epoch 199/200: 100%|██████████| 63/63 [03:41<00:00,  3.51s/it]


Epoch 199: Average Loss = 3.8684


Epoch 200/200: 100%|██████████| 63/63 [03:41<00:00,  3.52s/it]

Epoch 200: Average Loss = 3.8682


In [ ]:
class FlowerDataset(Dataset):
  def __init__(self, path, indices, labels, transform=None):
    self.path = path
    self.indices = indices
    self.labels = labels
    self.transform = transform
    self.samples = [
        (f"image_{idx:05d}.jpg", labels[idx - 1])
        for idx in indices
    ]

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    image_name, label = self.samples[idx]
    image_path = os.path.join(self.path, image_name)
    image = Image.open(image_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label - 1

In [ ]:
import scipy.io

label = scipy.io.loadmat("/content/drive/MyDrive/hw3/imagelabels.mat") #change to the path where your dataset is in
setid = scipy.io.loadmat("/content/drive/MyDrive/hw3/setid.mat") #change to the path where your dataset is in

all_labels = label["labels"][0] #flattening it to 1D array

train_idx = setid["trnid"][0]
valid_idx = setid["valid"][0]
test_idx = setid["tstid"][0]

In [ ]:
evaluation_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data =FlowerDataset("/content/drive/MyDrive/hw3/jpg", train_idx, all_labels, transform = evaluation_transform) #change to the path where your dataset is in
valid_data =FlowerDataset("/content/drive/MyDrive/hw3/jpg", valid_idx, all_labels, transform = evaluation_transform) #change to the path where your dataset is in
test_data =FlowerDataset("/content/drive/MyDrive/hw3/jpg", test_idx, all_labels, transform = evaluation_transform) #change to the path where your dataset is in

In [ ]:
for p in model.encoder.parameters():
  p.requires_grad = False #freeze the encoder

NameError: name 'model' is not defined

In [ ]:
class LinearClassifier(nn.Module):
  def __init__(self, input_dim=512, class_number=102):
    super().__init__()
    self.fc = nn.Linear(input_dim, class_number)

  def forward(self, x):
    return self.fc(x)

linear_model = LinearClassifier(input_dim=512).to(device)

NameError: name 'device' is not defined

In [ ]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

def extract_features(dataloader, model):
  model.eval()
  features, labels = [], [] #create lists to store features and labels
  with torch.no_grad():
    for image, label in dataloader:
      image = image.to(device)
      output = model.encoder(image)
      output = model.flatten(output)
      features.append(output.cpu()) #move it to CPU to save GPU space
      labels.append(label)
  return torch.cat(features), torch.cat(labels) #concatenate all batch features and labels into single tensors

#few shot
train_features, train_labels = extract_features(train_loader, model)
valid_features, valid_labels = extract_features(valid_loader, model)
test_features,  test_labels  = extract_features(test_loader, model)

NameError: name 'model' is not defined

In [ ]:
from torch.utils.data import TensorDataset
def evaluate_classifier(model_class, train_features, train_labels,
                        valid_features, valid_labels,
                        test_features, test_labels,
                        input_dim=512, class_number=102, hidden=False,
                        epochs=100, batch_size=128, lr=1e-3, device="cuda"):

    if hidden:
        classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, class_number)
        )
    else:
        classifier = nn.Linear(input_dim, class_number)

    classifier = classifier.to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    train_loader = DataLoader(TensorDataset(train_features, train_labels), batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(TensorDataset(valid_features, valid_labels), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(TensorDataset(test_features,  test_labels),  batch_size=batch_size, shuffle=False)

    classifier.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = classifier(x)
            loss = loss_fn(output, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[Epoch {epoch:03}] Train Loss = {total_loss:.4f}")

    def eval(loader, name):
        classifier.eval()
        total_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                output = classifier(x)
                loss = loss_fn(output, y)
                total_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                correct += (predicted == y).sum().item()
                total += y.size(0)
        acc = 100 * correct / total
        avg_loss = total_loss / len(loader)
        print(f"{name} Loss = {avg_loss:.4f}, Accuracy = {acc:.2f}%")

    eval(valid_loader, "Validation")
    eval(test_loader,  "Test")

In [ ]:
# Linear Probe
evaluate_classifier(
    model_class=LinearClassifier,
    train_features=train_features, train_labels=train_labels,
    valid_features=valid_features, valid_labels=valid_labels,
    test_features=test_features,   test_labels=test_labels,
    hidden=False  # Linear
)

# MLP
evaluate_classifier(
    model_class=None,
    train_features=train_features, train_labels=train_labels,
    valid_features=valid_features, valid_labels=valid_labels,
    test_features=test_features,   test_labels=test_labels,
    hidden=True  # MLP
)

[Epoch 001] Train Loss = 35.2226
[Epoch 002] Train Loss = 27.3957
[Epoch 003] Train Loss = 21.7441
[Epoch 004] Train Loss = 17.7223
[Epoch 005] Train Loss = 14.8738
[Epoch 006] Train Loss = 12.8700
[Epoch 007] Train Loss = 11.3842
[Epoch 008] Train Loss = 10.2810
[Epoch 009] Train Loss = 9.4009
[Epoch 010] Train Loss = 8.6729
[Epoch 011] Train Loss = 8.0961
[Epoch 012] Train Loss = 7.5879
[Epoch 013] Train Loss = 7.1547
[Epoch 014] Train Loss = 6.7692
[Epoch 015] Train Loss = 6.3855
[Epoch 016] Train Loss = 6.1209
[Epoch 017] Train Loss = 5.8132
[Epoch 018] Train Loss = 5.5519
[Epoch 019] Train Loss = 5.3048
[Epoch 020] Train Loss = 5.0973
[Epoch 021] Train Loss = 4.9179
[Epoch 022] Train Loss = 4.7218
[Epoch 023] Train Loss = 4.5386
[Epoch 024] Train Loss = 4.3725
[Epoch 025] Train Loss = 4.2250
[Epoch 026] Train Loss = 4.0689
[Epoch 027] Train Loss = 3.9280
[Epoch 028] Train Loss = 3.8134
[Epoch 029] Train Loss = 3.6804
[Epoch 030] Train Loss = 3.5668
[Epoch 031] Train Loss = 3.4606


#MAE

In [ ]:
import torch
import torch.nn as nn

class Masking(nn.Module):
  def __init__(self, image_size=224, patch_size=16, input_channels=3, embedding_dim=512, mask_ratio=0.5):
    super().__init__()
    self.image_size = image_size
    self.patch_size = patch_size
    self.patch_number = (image_size // patch_size) ** 2
    self.input_channels = input_channels
    self.embedding_dim = embedding_dim
    self.mask_ratio = mask_ratio
    self.patch_dim = patch_size * patch_size * input_channels
    self.project = nn.Linear(self.patch_dim, embedding_dim)

  def forward(self, images):
    batch, channel, height, width = images.shape #shape unpack
    assert height == self.image_size and width == self.image_size, "Image size mismatch"
    patches = images.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size) #define non-overlapping patches(chop height and width based on patch size) -> (batch, channel, patch_number_w, patch_number_h, patch_size, patch_size)
    patches = patches.permute(0, 2, 3, 1, 4, 5)
    patches = patches.reshape(batch, self.patch_number, self.patch_dim)
    patches = self.project(patches) #(batch, patch_number, embedding_dim)
    patch_number = patches.shape[1]
    visible_patch_number = int(patch_number * (1 - self.mask_ratio))

    noise = torch.rand(batch, patch_number, device=images.device)
    idx_sort = torch.argsort(noise, dim=1) #showing the index in ascending order(the index with smallest value goes to position 0 and so on)
    idx_original = torch.argsort(idx_sort, dim=1)
    idx_visible = idx_sort[:, :visible_patch_number] #unmasked
    visible_patches = torch.gather(patches, dim=1, index = idx_visible.unsqueeze(-1).repeat(1, 1, self.embedding_dim))
    mask = torch.ones([batch, patch_number], device=images.device)
    mask[:, :visible_patch_number] = 0 #Ex. [0, 0, 1, 1]
    mask = torch.gather(mask,dim=1, index=idx_original) #Ex.[0, 1, 0, 1], so now we now which patch is kept based on their original order
    return visible_patches, mask, idx_original

In [ ]:
class Encoder(nn.Module):
  def __init__(self, embedding_dim=512, hidden_dim=1024, layer_number=4):
    super().__init__()
    encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, dim_feedforward=hidden_dim, nhead=8, batch_first=True) #define one layer
    self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layer_number) #replicate the layer to form the full encoder

  def forward(self, x):
    return self.encoder(x)

In [ ]:
class Decoder(nn.Module):
  def __init__(self, embedding_dim=512, decoder_dim=512, hidden_dim=1024, patch_number=196):
    super().__init__()
    self.patch_number = patch_number
    self.mask_token = nn.Parameter(torch.randn(1, 1, decoder_dim)) #fill mask token in masked position for reconstruction (for 1 batch, 1 token)
    self.decoder_embedding = nn.Linear(embedding_dim, decoder_dim) #project encoder output to decoder dimension
    decoder_layer = nn.TransformerEncoderLayer(d_model=decoder_dim, dim_feedforward=hidden_dim, nhead=8, batch_first=True)
    self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=4)
    self.decoder_project = nn.Linear(decoder_dim, embedding_dim)

  def forward(self, x, idx_original):
    x = self.decoder_embedding(x)
    masked_number = self.patch_number - x.shape[1]
    mask_tokens = self.mask_token.repeat(x.shape[0], masked_number, 1)
    x_all = torch.cat([x, mask_tokens], dim=1)
    x_all = torch.gather(x_all, dim=1, index=idx_original.unsqueeze(-1).repeat(1, 1, x_all.shape[2]))
    x = self.decoder(x_all)
    x = self.decoder_project(x)

    return x

In [ ]:
def MAE_loss(predicted, target, mask): #only for masked patches
  mask = mask.unsqueeze(-1).type_as(predicted)
  loss = ((predicted - target) ** 2 * mask).sum() / mask.sum()
  return loss

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
masking = Masking(image_size=224, patch_size=16, input_channels=3, embedding_dim=512, mask_ratio=0.75)
encoder = Encoder(embedding_dim=512, hidden_dim=1024, layer_number=8)
decoder = Decoder(embedding_dim=512, decoder_dim=512, hidden_dim=1024, patch_number=196)
masking.to(device)
encoder.to(device)
decoder.to(device)

optimizer = torch.optim.Adam(
    list(masking.parameters()) +
    list(encoder.parameters()) +
    list(decoder.parameters()),
    lr=5e-5
)

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Define transform (resize to 224 since patching expects it)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(root="/content/drive/MyDrive/hw3", transform=transform) #change to the path where your dataset is in
train_loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

epochs = 200

for epoch in range(1, epochs + 1):
    total_loss = 0 #accumulates total loss over all batches in each epoch
    masking.train()
    encoder.train()
    decoder.train()

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")

    for images, _ in loop: #ignore labels
        images = images.to(device)

        visible_patches, mask, idx_original = masking(images)

        with torch.no_grad():
          patches = images.unfold(2, masking.patch_size, masking.patch_size).unfold(3, masking.patch_size, masking.patch_size)
          patches = patches.permute(0, 2, 3, 1, 4, 5)
          patches = patches.reshape(images.size(0), masking.patch_number, masking.patch_dim)
          all_patches = masking.project(patches)
          all_patches = torch.gather(all_patches, dim=1, index=idx_original.unsqueeze(-1).repeat(1, 1, all_patches.shape[2]))

        encoded = encoder(visible_patches) #encoding
        reconstructed = decoder(encoded, idx_original) #decoding
        loss = MAE_loss(reconstructed, all_patches, mask) #compute loss only on masked patches

        optimizer.zero_grad() #clear previous gradient
        loss.backward() #compute gradient by backpropagation
        optimizer.step() #update

        total_loss += loss.item() #accumulative loss
        loop.set_postfix(loss=loss.item())

    print(f"[Epoch {epoch}] MAE Loss = {total_loss:.4f}")

Epoch 1/200: 100%|██████████| 127/127 [01:55<00:00,  1.10it/s, loss=55.6]


[Epoch 1] MAE Loss = 7973.3030


Epoch 2/200: 100%|██████████| 127/127 [01:09<00:00,  1.82it/s, loss=47]


[Epoch 2] MAE Loss = 6193.7131


Epoch 3/200: 100%|██████████| 127/127 [01:09<00:00,  1.82it/s, loss=48.4]


...

[Epoch 198] MAE Loss = 5120.6649


Epoch 199/200: 100%|██████████| 127/127 [01:09<00:00,  1.83it/s, loss=43.7]


[Epoch 199] MAE Loss = 5179.0005


Epoch 200/200: 100%|██████████| 127/127 [01:09<00:00,  1.84it/s, loss=40.7]

[Epoch 200] MAE Loss = 5155.4960


In [ ]:
for p in encoder.parameters():
  p.requires_grad = False

In [ ]:
def extract_features_MAE(dataloader, model):
  model.eval()
  features, labels = [], []
  with torch.no_grad():
    for image, label in dataloader:
      image = image.to(device)
      visible_patches, _, _ = masking(image)
      output = model(visible_patches)
      output = output.mean(dim=1)
      features.append(output.cpu())
      labels.append(label)
  return torch.cat(features), torch.cat(labels)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

#few shot
train_features_MAE, train_labels_MAE = extract_features_MAE(train_loader, encoder)
valid_features_MAE, valid_labels_MAE = extract_features_MAE(valid_loader, encoder)
test_features_MAE, test_labels_MAE = extract_features_MAE(test_loader, encoder)

In [ ]:
evaluate_classifier(
    model_class=LinearClassifier,
    train_features=train_features_MAE, train_labels=train_labels_MAE,
    valid_features=valid_features_MAE, valid_labels=valid_labels_MAE,
    test_features=test_features_MAE,   test_labels=test_labels_MAE,
    hidden=False
)

evaluate_classifier(
    model_class=None,
    train_features=train_features_MAE, train_labels=train_labels_MAE,
    valid_features=valid_features_MAE, valid_labels=valid_labels_MAE,
    test_features=test_features_MAE,   test_labels=test_labels_MAE,
    hidden=True
)

[Epoch 001] Train Loss = 37.2699
[Epoch 002] Train Loss = 34.7146
[Epoch 003] Train Loss = 33.0800
[Epoch 004] Train Loss = 32.1073
[Epoch 005] Train Loss = 31.3226
[Epoch 006] Train Loss = 30.7947
[Epoch 007] Train Loss = 30.3371
[Epoch 008] Train Loss = 29.9970
[Epoch 009] Train Loss = 29.6903
[Epoch 010] Train Loss = 29.4191
[Epoch 011] Train Loss = 29.1702
[Epoch 012] Train Loss = 28.9648
[Epoch 013] Train Loss = 28.7847
[Epoch 014] Train Loss = 28.6851
[Epoch 015] Train Loss = 28.4811
[Epoch 016] Train Loss = 28.2950
[Epoch 017] Train Loss = 28.1780
[Epoch 018] Train Loss = 28.0301
[Epoch 019] Train Loss = 27.9140
[Epoch 020] Train Loss = 27.7644
[Epoch 021] Train Loss = 27.6157
[Epoch 022] Train Loss = 27.5441
[Epoch 023] Train Loss = 27.4199
[Epoch 024] Train Loss = 27.2695
[Epoch 025] Train Loss = 27.2042
[Epoch 026] Train Loss = 27.1259
[Epoch 027] Train Loss = 27.0081
[Epoch 028] Train Loss = 26.9028
[Epoch 029] Train Loss = 26.7873
[Epoch 030] Train Loss = 26.6561
[Epoch 031